In [1]:
from pulp import LpMaximize , LpProblem , LpVariable , lpSum ,LpBinary, PULP_CBC_CMD
import random
import time
import numpy as np
import pandas as pd

In [30]:
limits = {"provider1": 2, "provider2": 200, "provider777":12}
prizes = {"user1":100, "user123124":333}

#pd.Series(W).unstack().loc["user1"].reindex(limits.keys())#.isna().sum()

input_dataframe = pd.DataFrame([
    ["user1", "credential1", "provider1", 1], 
    ["user1", "credential2", "provider1", 1], 
    ["user1", "credential3", "provider2", 1], 
    ["user3", "credential4", "provider3", 1], 
    ["user4", "credential5", "provider3", 1],
    ["user5", "credential6", "provider1", 1],
    ], columns = ["user", "credential", "provider", "credential_weight"])

assert input_dataframe["credential"].duplicated().sum() == 0
# assert pd.Series(user_to_credentials).explode().duplicated().sum()==0, "Some credetnaials are duplicated"
# assert pd.Series(provider_to_credentials).explode().duplicated().sum()==0, "Some credetnaials are duplicated"

input_dataframe = input_dataframe.set_index("credential")

W = input_dataframe.groupby(["user", "provider"])["credential_weight"].sum().unstack(fill_value=0)
users, providers = W.index, W.columns
credentials = input_dataframe.index.tolist()
prizes = {user: prizes.get(user, 1) for user in users} # if user is not in prizes, then prize is 1
limits = {provider: limits.get(provider, float("inf")) for provider in providers} # filter out providers that are not in the data
creds_weights = input_dataframe["credential_weight"].to_dict()

In [31]:
# Define the problem
prob = LpProblem("ILP_Problem", LpMaximize)

In [32]:
user_to_credentials = input_dataframe.groupby("user").apply(lambda x: set(x.index)).to_dict()
provider_to_credentials = input_dataframe.groupby("provider").apply(lambda x: set(x.index)).to_dict()

In [33]:
# Calculate the start time
start = time.time()

In [34]:
# Variables
x = LpVariable.dicts("x", users , cat = LpBinary) # Binary variable for family selection
y = LpVariable.dicts("y", [(credential ,provider) for credential in credentials for provider in providers] ,cat = LpBinary) # Binary variable for item assignment

# Objective Function
prob += lpSum(prizes[user] * x [user] for user in users)

In [38]:
#Constraints

# The sum of the weights of the credentials assigned to a provider must be less than or equal to the limit of the provider
for provider, limit in limits.items():
    if limit != float("inf"):
        prob += lpSum(creds_weights[credential] * y[credential , provider] for credential in credentials) <= limits[provider]
        
# The sum of the weights of the credentials assigned to a user must be less than or equal to the limit of the user
for user, user_credentials in user_to_credentials.items():
    for credential in user_credentials:
        prob += lpSum (y[credential, provider] for provider in providers) == x[user]


# The credentials that are not allowed to be assigned to a provider must be 0

set_credentials = set(credentials)
for provider in providers:
    misalowed_credentials = set_credentials - provider_to_credentials[provider]
    for credential in misalowed_credentials:
        prob += y[credential , provider] == 0


# Solve the problem
prob.solve(PULP_CBC_CMD(msg=True)) #
#prob . solve ()

# Calculate the end time and time taken
end = time.time()
length = end - start

pd.Series(x).map(lambda x: x.varValue)
pd.Series(y).unstack().map(lambda x: x.varValue)

user1    1.0
user3    1.0
user4    1.0
user5    0.0
dtype: float64

In [28]:
flie_name = f'fn = {fn} bn = {bn} In = {In}.txt'
file1 = open(flie_name, 'w')
file1.write(f'time of runing = {length}')

# Print the results
print (" Objective value :", prob . objective . value () )
for v in prob . variables () :
    txt = f'{v . name} = {v . varValue}'
    file1.write(txt)

 Objective value : 3.0
